In [ ]:
import xarray as xr
import numpy as np

# Load ERA5 to take land_sea_mask and geopotential at surface from dataset
era5_dir="/hkfs/work/workspace/scratch/xo8179-neural_lam/data/data/global_era5_1980_2022_6h-128x64_equiangular_with_poles_conservative/fields.zarr"
ds = xr.open_dataset(era5_dir)
print(ds)

<xarray.Dataset> Size: 167GB
Dimensions:                  (time: 61400, longitude: 128, latitude: 64,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 491kB 1979-12-23 ... 2021-...
Data variables: (12/13)
    10m_u_component_of_wind  (time, longitude, latitude) float32 2GB ...
    10m_v_component_of_wind  (time, longitude, latitude) float32 2GB ...
    2m_temperature           (time, longitude, latitude) float32 2GB ...
    geopotential             (time, level, longitude, latitude) float32 26GB ...
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...
    ...                       ...
    specific_humidity        

In [6]:
import var_dicts as d
from pathlib import Path

# Atmospheric variables to process
atm_vars = ["t", "u", "v", "w", "z", "q"]

# Surface variables
surface_vars = ["2t", "10u", "10v", "msl"]

# Your working directory
data_dir = Path("/hkfs/work/workspace/scratch/xo8179-neural_lam/data/data/global_nextgems_2046_2049_equiangular_with_poles_conservative/")

out_name = "fields.zarr"

t_chunk = 1

In [3]:
# Open and concatenate atmospheric variables
atm_datasets = []

for var in atm_vars:
    
    
    # Open as xarray datasets
    ds_var_decade = xr.open_zarr(str(data_dir / f"3D_nextgems_128x64_2046_2049_6hourly.nc_{var}.zarr"))

    ds_var_decade = ds_var_decade.astype(np.float32)
        
    var_rename_dict_filtered = {
        k: v for k, v in d.var_rename_dict.items()
        if k in ds_var_decade.variables or k in ds_var_decade.dims
    }
    ds_var_decade = ds_var_decade.rename(var_rename_dict_filtered)
    
    
    atm_datasets.append(ds_var_decade)
    # print(ds_concat)

# Merge all atmospheric variables into one dataset
ds_atm = xr.merge(atm_datasets)

# ds_atm = ds_atm.rename(var_rename_dict)



In [4]:
print(ds_atm)

<xarray.Dataset> Size: 15GB
Dimensions:              (time: 5844, level: 13, longitude: 128, latitude: 64)
Coordinates:
  * latitude             (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                (level) int64 104B 50 100 150 200 ... 700 850 925 1000
  * longitude            (longitude) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * time                 (time) datetime64[ns] 47kB 2046-01-01 ... 2049-12-31...
Data variables:
    temperature          (time, level, longitude, latitude) float32 2GB dask.array<chunksize=(1, 13, 128, 64), meta=np.ndarray>
    u_component_of_wind  (time, level, longitude, latitude) float32 2GB dask.array<chunksize=(1, 13, 128, 64), meta=np.ndarray>
    v_component_of_wind  (time, level, longitude, latitude) float32 2GB dask.array<chunksize=(1, 13, 128, 64), meta=np.ndarray>
    vertical_velocity    (time, level, longitude, latitude) float32 2GB dask.array<chunksize=(1, 13, 128, 64), meta=np.ndarray>
    geopotential         (time, l

In [5]:
ds_atm = ds_atm.chunk({"time": t_chunk, "level": 13, "longitude": 128, "latitude": 64})
encoding = {var: {"chunks": (t_chunk, 13, 128, 64)} for var in ds_atm.data_vars}
ds_atm.to_zarr(data_dir.joinpath(out_name), mode="w", encoding=encoding)

In [5]:
print(ds_atm['time'][-1])

<xarray.DataArray 'time' ()> Size: 8B
array('2049-12-31T18:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    time     datetime64[ns] 8B 2049-12-31T18:00:00


In [7]:
surface_datasets = []

for var in surface_vars:
    # Open as xarray datasets
    ds_var_surface = xr.open_zarr(str(data_dir / f"2D_nextgems_128x64_2046_2049_6hourly.nc_{var}.zarr"))

    ds_var_surface = ds_var_surface.astype(np.float32)
        
    var_rename_dict_filtered = {
        k: v for k, v in d.var_rename_dict.items()
        if k in ds_var_surface.variables or k in ds_var_surface.dims
    }
    ds_var_surface = ds_var_surface.rename(var_rename_dict_filtered)
    surface_datasets.append(ds_var_surface)

for dataset in surface_datasets:
    print(dataset)
# Merge all surface variables into one dataset
ds_surface = xr.merge(surface_datasets)

# ds_surface = ds_surface.rename(var_rename_dict)

# Apply atmospheric chunking
ds_surface = ds_surface.chunk({"time": t_chunk, "longitude": 128, "latitude": 64})


encoding = {var: {"chunks": (t_chunk, 128, 64)} for var in ds_surface.data_vars}
ds_surface.to_zarr(data_dir.joinpath(out_name), mode="a", encoding=encoding)

<xarray.Dataset> Size: 192MB
Dimensions:         (time: 5844, longitude: 128, latitude: 64)
Coordinates:
  * latitude        (latitude) float64 512B -90.0 -87.14 -84.29 ... 87.14 90.0
  * longitude       (longitude) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * time            (time) datetime64[ns] 47kB 2046-01-01 ... 2049-12-31T18:0...
Data variables:
    2m_temperature  (time, longitude, latitude) float32 191MB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>
<xarray.Dataset> Size: 192MB
Dimensions:                  (time: 5844, longitude: 128, latitude: 64)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 47kB 2046-01-01 ... 2049-1...
Data variables:
    10m_u_component_of_wind  (time, longitude, latitude) float32 191MB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>
<xarray.Dataset> Size: 192MB
Dimens

In [8]:
# Open as xarray datasets
ds_var_tp = xr.open_zarr(str(data_dir / f"2D_nextgems_128x64_2046_2049_tp_6hourly.nc_tp.zarr"))

ds_var_tp = ds_var_tp.astype(np.float32)
    
var_rename_dict_filtered = {
    k: v for k, v in d.var_rename_dict.items()
    if k in ds_var_tp.variables or k in ds_var_tp.dims
}
ds_var_tp = ds_var_tp.rename(var_rename_dict_filtered)

print(ds_var_tp)

# Apply atmospheric chunking
ds_ds_var_tpsurface = ds_var_tp.chunk({"time": t_chunk, "longitude": 128, "latitude": 64})


encoding = {var: {"chunks": (t_chunk, 128, 64)} for var in ds_var_tp.data_vars}
ds_var_tp.to_zarr(data_dir.joinpath(out_name), mode="a", encoding=encoding)

<xarray.Dataset> Size: 192MB
Dimensions:                  (time: 5844, longitude: 128, latitude: 64)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 47kB 2046-01-01 ... 2049-1...
Data variables:
    total_precipitation_6hr  (time, longitude, latitude) float32 191MB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>


In [9]:

ds_era5 = xr.open_dataset(era5_dir)
remaining = ds_era5[["land_sea_mask", "geopotential_at_surface"]]

remaining.to_zarr(data_dir.joinpath(out_name), mode="a")


In [10]:
ds = xr.open_dataset(data_dir.joinpath("fields.zarr"))
print(ds)

<xarray.Dataset> Size: 16GB
Dimensions:                  (time: 5844, longitude: 128, latitude: 64,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 47kB 2046-01-01 ... 2049-1...
Data variables: (12/13)
    10m_u_component_of_wind  (time, longitude, latitude) float32 191MB ...
    10m_v_component_of_wind  (time, longitude, latitude) float32 191MB ...
    2m_temperature           (time, longitude, latitude) float32 191MB ...
    geopotential             (time, level, longitude, latitude) float32 2GB ...
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...
    ...                       ...
    specific_humidity     